In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: The domain policy has disabled Drive File Stream: https://support.google.com/a/answer/7496409

In [1]:
!pip install pytesseract

In [2]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from torchvision import models, transforms
import torch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import pytesseract

# For downloading images
import os

# For handling warnings
import warnings
warnings.filterwarnings("ignore")


In [3]:
from PIL import Image, UnidentifiedImageError
def download_images(df, output_dir="images"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for index, row in df.iterrows():
        image_url = row['image_link']
        try:
            response = requests.get(image_url)
            # Check if the response contains image content
            if 'image' not in response.headers.get('content-type', ''):
                print(f"URL at index {index} is not an image. Skipping.")
                continue

            img = Image.open(BytesIO(response.content))
            img.save(f"{output_dir}/{index}.jpg")
        except UnidentifiedImageError:
            print(f"Error: Unable to identify image at index {index} with URL {image_url}")
        except Exception as e:
            print(f"Failed to download or process image at index {index}. Error: {e}")

df = pd.read_csv("dataset/train.csv")
download_images(df)



URL at index 520 is not an image. Skipping.
Failed to download or process image at index 1096. Error: image file is truncated (2 bytes not processed)
URL at index 7754 is not an image. Skipping.


KeyboardInterrupt: 

In [12]:
import shutil

# Path to the images folder
folder_to_zip = "images"
output_zip = "images.zip"

# Compress the images folder into a zip file
shutil.make_archive(output_zip.replace(".zip", ""), 'zip', folder_to_zip)


KeyboardInterrupt: 

In [11]:
from google.colab import files

# Download the zip file
files.download("images.zip")


FileNotFoundError: Cannot find file: images.zip

In [6]:
# Preprocessing function for images
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Use a pre-trained ResNet model for feature extraction
model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*(list(model.children())[:-1]))  # Remove the classification layer
model.eval()  # Set to evaluation mode

def extract_image_features(image_path):
    img = Image.open(image_path).convert('RGB')
    img_tensor = preprocess(img).unsqueeze(0)  # Add batch dimension
    with torch.no_grad():
        features = model(img_tensor).flatten().numpy()  # Extract image features
    return features


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 154MB/s]


In [7]:
def extract_text_from_image(image_path):
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    return text


In [9]:
!sudo apt update
!sudo apt install -y tesseract-ocr


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [3,030 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 https://r2u.st

In [19]:
import os
import pandas as pd
import pytesseract
from PIL import Image, UnidentifiedImageError
features = []

def extract_features_and_text(df, image_dir="images"):


    for index, row in df.iterrows():
        image_path = f"{image_dir}/{index}.jpg"
        if not os.path.isfile(image_path):
            print(f"Image file at index {index} does not exist. Skipping.")
            features.append({'index': index, 'text': "", 'features': {}})
            continue

        try:
            img = Image.open(image_path)
            text = pytesseract.image_to_string(img)
            # Example feature extraction logic; adapt as needed
            features_dict = {'text': text}  # Customize this based on what features you need

            features.append({'index': index, 'text': text, 'features': features_dict})
        except UnidentifiedImageError:
            print(f"Error: Unable to identify image at index {index} with path {image_path}")
            features.append({'index': index, 'text': "", 'features': {}})
        except Exception as e:
            print(f"Failed to process image at index {index}. Error: {e}")
            features.append({'index': index, 'text': "", 'features': {}})

    return features

# Example usage
df = pd.read_csv("dataset/train.csv")
features = extract_features_and_text(df)


KeyboardInterrupt: 

In [22]:
print(features[:5])  # Print the first few entries in the features list


[{'index': 0, 'text': '100% NATUR\n\n \n\x0c', 'features': {'text': '100% NATUR\n\n \n\x0c'}}, {'index': 1, 'text': '  \n  \n   \n \n\nLEBENSMITTELECHT\n\ngsi\n\n \n\nGEPRAGTES\n\npar Site lp te\n\nea UND GESCHUTZTE DESIGNS\n\x0c', 'features': {'text': '  \n  \n   \n \n\nLEBENSMITTELECHT\n\ngsi\n\n \n\nGEPRAGTES\n\npar Site lp te\n\nea UND GESCHUTZTE DESIGNS\n\x0c'}}, {'index': 2, 'text': 'Serving Size: 1 Tablet (0.709 g) | Each serving contains (Approx. Values):\n\nIngredient Oty, / Serving\n\n \n\n*PHOSPHOcomplex® Silybin (Sillybum marianum) 200 mg\nDandelion (Taraxacum officinale) leaf extract - 10:1 100 mg\nKutki (Picrorhiza kurroa)rhizome extract - 0.5% Bitters 50 mg\nKasani (Cichorium intybus) seed extract - 1% Bitters 25 mg\nPunarnava (Boerhavia diffusa) root extract - 0.07% alkaloids 25 mg\nBhui amla (Phyllanthus amarus) WP extract - 0.5% Bitters 25 mg\nAmla (Phyllanthus emblica) fruit extract - 10% Tannins 25 mg\nLicorice (Glycyrrhiza glabra) root extract - 5% Glycyrrhizin 25 

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import numpy as np

In [36]:
# Extract features and target values
# features = extract_features_and_text(df)
texts = [f['text'] for f in features]
vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(texts)

# Extract numerical features from the `features` dictionary
# Assuming `features` contains numerical features; if not, handle accordingly
# For demonstration, let's create dummy image features (replace with actual features extraction)
X_image = np.random.rand(len(features), 10)  # Example: 10-dimensional feature vectors for images

# Combine text features and image features
X = np.hstack([X_text.toarray(), X_image])  # Combine text and image features

# Assuming `entity_value` is your target in the DataFrame
# For example purpose, we'll simulate `y` as random values (replace with actual target values)
y = np.random.rand(len(features))

# Ensure target values are numeric (for regression)
# y = df['entity_value']  # Replace with actual target extraction

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_train)

# Make predictions
y_pred = regressor.predict(X_test)


In [32]:
# Combine image features and categorical data
X = [f['text'] for f in features]  # Image features
y = df['entity_value'][:277] # Target values (e.g., weight, volume, etc.)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Use a Random Forest Regressor for the prediction
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_train)


ValueError: could not convert string to float: 'ROLL SHADE\n\n220GSM PERMEABLE FABRIC\n\nA PROFESSIONAL\n\nwi HAND MADE\n\n| DURABLE\n\n \n\n \n\n \n\x0c'

In [35]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Function to extract features from text (example)
def extract_text_features(text, vectorizer):
    return vectorizer.transform([text]).toarray()

# Example data
df = pd.read_csv("dataset/train.csv")

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
text_features = vectorizer.fit_transform(df['text'])

# Example image features extraction (assuming they are precomputed)
# Replace with actual image feature extraction
image_features = np.random.rand(text_features.shape[0], 10)  # Dummy features

# Combine text and image features
X = np.hstack([text_features.toarray(), image_features])
y = df['entity_value']

# Convert target values to numeric if necessary
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_train)

# Make predictions (example)
y_pred = regressor.predict(X_test)


KeyError: 'text'

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Load the TF-IDF vectorizer fitted on the training data
vectorizer = TfidfVectorizer()  # Replace with your trained vectorizer

def extract_image_features(image_path):
    # Implement your image feature extraction here
    # For example, using a pre-trained CNN model
    return np.random.rand(10)  # Dummy implementation; replace with actual features

def predict_and_format_output(test_df, regressor, vectorizer, image_dir="images"):
    predictions = []
    for index, row in test_df.iterrows():
        # Extract text features from the test set
        text_features = vectorizer.transform([row['text']])

        # Extract image features
        image_path = os.path.join(image_dir, f"{index}.jpg")
        image_features = extract_image_features(image_path)

        # Ensure text and image features are combined correctly
        test_features = np.hstack([text_features.toarray(), image_features])

        # Predict entity value
        predicted_value = regressor.predict([test_features])[0]

        # Format the output as "x unit"
        entity_name = row['entity_name']
        unit = "gram"  # You will need to map the entity_name to the correct unit from the entity_unit_map
        formatted_prediction = f"{predicted_value:.2f} {unit}"
        predictions.append((index, formatted_prediction))

    return pd.DataFrame(predictions, columns=["index", "prediction"])

# Load test dataset
test_df = pd.read_csv("dataset/test.csv")

# Ensure the vectorizer used for prediction matches the one used for training
vectorizer.fit(df['text'])  # Fit on training text data

# Make predictions
output_df = predict_and_format_output(test_df, regressor, vectorizer)

# Save to CSV for submission
output_df.to_csv("test_out.csv", index=False)


KeyError: 'text'

In [39]:
def predict_and_format_output(test_df, regressor, image_dir="images"):
    predictions = []
    for index, row in test_df.iterrows():
        image_path = os.path.join(image_dir, f"{index}.jpg")
        image_features = extract_image_features(image_path)

        # Predict entity value
        predicted_value = regressor.predict([image_features])[0]

        # Format the output as "x unit"
        entity_name = row['entity_name']
        unit = "gram"  # You will need to map the entity_name to the correct unit from the entity_unit_map
        formatted_prediction = f"{predicted_value:.2f} {unit}"
        predictions.append((index, formatted_prediction))

    return pd.DataFrame(predictions, columns=["index", "prediction"])

# Load test dataset
test_df = pd.read_csv("dataset/test.csv")

# Make predictions
output_df = predict_and_format_output(test_df, regressor)

# Save to CSV for submission
output_df.to_csv("test_out.csv", index=False)


ValueError: X has 10 features, but RandomForestRegressor is expecting 3998 features as input.

In [ ]:
python src/sanity.py --output test_out.csv
